# Jupyter notebook workflow: create, populate, run, test, and export
This notebook documents a reproducible workflow for creating, populating, executing, testing, and exporting Jupyter notebooks programmatically and via CLI/VSCode.

## 1) Project and virtual environment (CLI)
Create a project folder and virtual environment on Windows:

```
python -m venv .venv
.venv\Scripts\activate
```

Create `requirements.txt` and add pinned dependencies.

In [ ]:
# Create a minimal requirements.txt file programmatically
requirements = ['requests==2.31.0','beautifulsoup4==4.12.2','pandas==2.2.2','matplotlib==3.8.2','nbformat==5.8.0','nbconvert==7.2.9']
with open('requirements.txt','w') as f:
    f.write('\n'.join(requirements))
print('Wrote requirements.txt with', len(requirements), 'packages')

## 2) Install Jupyter and dependencies (pip)
Install packages (in the activated virtual environment):

```
pip install jupyterlab notebook nbformat nbconvert pytest pandas matplotlib requests beautifulsoup4
```

You can also run `pip install -r requirements.txt`.

## 3) Programmatically create a .ipynb (nbformat)
Example: build a notebook object and write to disk using `nbformat`.

In [ ]:
import nbformat
nb = nbformat.v4.new_notebook()
nb.cells = [nbformat.v4.new_markdown_cell('# Auto-generated notebook'),
            nbformat.v4.new_code_cell("print('Hello from generated notebook')")]
nbformat.write(nb, 'generated_notebook.ipynb')
print('Wrote generated_notebook.ipynb')

## 4) Add code cells: functions, data, plots
Below is a short example that defines a function, creates a DataFrame, and plots it with matplotlib.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def myfunc(x):
    return x*2

# sample data
DF = pd.DataFrame({'x': range(5), 'y': [myfunc(i) for i in range(5)]})
print(DF)

# plot
plt.figure()
plt.plot(DF['x'], DF['y'], marker='o')
plt.title('Sample plot')
plt.show()

## 5) Add markdown cells and metadata (nbformat)
You can set notebook metadata such as the kernel and language before writing the file with nbformat.

In [ ]:
import nbformat
nb = nbformat.v4.new_notebook()
nb.metadata.kernelspec = {'name':'python3','display_name':'Python 3'}
nb.cells = [nbformat.v4.new_markdown_cell('# Example')]
nbformat.write(nb,'example_with_metadata.ipynb')
print('Wrote example_with_metadata.ipynb')

## 6) Execute the notebook programmatically (nbconvert ExecutePreprocessor)
Use ExecutePreprocessor to run a notebook and capture outputs programmatically.

In [ ]:
from nbconvert.preprocessors import ExecutePreprocessor
import nbformat

nb = nbformat.read('generated_notebook.ipynb', as_version=4)
ep = ExecutePreprocessor(timeout=600)
ep.preprocess(nb, {'metadata': {'path': '.'}})
nbformat.write(nb, 'generated_notebook_executed.ipynb')
print('Executed and wrote generated_notebook_executed.ipynb')

## 7) Write and run unit tests for notebook code (pytest)
Refactor reusable functions into a `.py` module and write simple pytest tests. Example:

```
# tests/test_myfunc.py
from mymodule import myfunc

def test_myfunc():
    assert myfunc(2) == 4
```

Run `pytest -q` to execute tests.

## 8) Export notebook to HTML/PDF (nbconvert)
Convert executed notebooks to HTML or PDF for sharing:

```
jupyter nbconvert --to html generated_notebook_executed.ipynb
# or
jupyter nbconvert --to pdf generated_notebook_executed.ipynb
```

## 9) Open and run in VSCode (CLI + run-cell commands)
Open the project in VSCode: `code .` and use the Notebook editor or the Python Interactive window. Right-click any cell -> Run Cell. Use the integrated terminal to run `jupyter lab` or `pytest`.

In [ ]:
# Practical scraping example: fetch page and save visible text
import requests
from bs4 import BeautifulSoup

url = "https://www.skysports.com/football/news/11095/13481245/world-cup-2026-fixture-schedule-and-uk-kick-off-times-day-by-day-breakdown-of-all-104-matches-including-england-scotland"
resp = requests.get(url)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')
page_text = soup.get_text(separator="\n", strip=True)
print('Title:', soup.title.string if soup.title else 'N/A')
print('\nFirst 500 chars:\n', page_text[:500])

# save
with open('page_text.txt','w',encoding='utf-8') as f:
    f.write(page_text)
print('\nSaved full text to page_text.txt')